# Workspace kickoff: Unity Catalog + source data

Run it **once on a new workspace** (or after wiping the data), cell by cell, from the top.

| Step | What it does | Time |
|---|---|---|
| 1 | Data generation parameters (catalog names live in `config/platform.yml`) | - |
| 2 | Repo paths, code import, loading `config/platform.yml` | seconds |
| 3 | SQL plan (dry-run): what will be created, nothing is executed | seconds |
| 4 | UC bootstrap: catalogs, schemas, volumes | ~1 min |
| 5 | UC verification | seconds |
| 6 | Smoke test: does writing Parquet to a volume work | seconds |
| 7 | Backfill: history 2025-01-01 → 2026-09-30 into landing | a few minutes |
| 8 | Landing verification: number of files per system/table | seconds |

**Requirements:**
- the notebook sits in the **repo root folder** (next to `uc_bootstrap.py`), in a Git folder in the Workspace,
- compute: **serverless**.

**Everything is safe to re-run:**
- the bootstrap uses `IF NOT EXISTS`,
- the backfill skips itself if landing already has data,
- the generator is deterministic, so a re-run produces identical files.

After a `Pull` in the Git folder (new simulator code), run **`%restart_python`** or Detach & re-attach. Python keeps once-imported modules in memory and will not see changes without a restart.

## 1. Configuration

Only the **data generation** parameters are here. Catalog, schema and source system names
are in `config/platform.yml` - the same file used by `uc_bootstrap.py`. The notebook derives the volume paths
(`LANDING`, `SMOKE`) from that file in step 2, so renaming the project means editing
a single line in the YAML. The rest of the notebook hardcodes nothing.

In [0]:
# History range to generate.
# 2025-01-01 is the export "go-live": on that day the CRM sends the initial customer snapshot.
# Starting later = no customers from before that date in the data.
BACKFILL_START = "2025-01-01"
BACKFILL_END = "2026-09-30"

# Data scale.
#   1.0  -> full scale, ~0.9 GB, a few minutes (for recordings)
#   0.2  -> quick run, < 1 min (tests, viewers with a limited quota)
SCALE = 1.0

# True = generate the backfill even if landing already has files (overwrites the same files).
# Use e.g. after changing SCALE. Note: when changing the scale, clear landing first,
# otherwise you will be left with a mix of old and new files from days outside the new range.
FORCE_BACKFILL = False

## 2. Repo paths and code import

A notebook run from a Git folder has its **working directory (`os.getcwd()`) = the folder containing the notebook**, i.e. the repo root folder.
So there is no need to type an e-mail or repo name: the same cell will work on any workspace and for any viewer.

Python imports modules only from folders listed in `sys.path`, so we add:
- `REPO`, so that `import uc_bootstrap` works,
- `REPO/retail-simulator`, so that `import simulator` works. The `simulator` package is a folder inside `retail-simulator`.

Finally, `load_config()` loads and validates `config/platform.yml`, and we derive the volume paths from it.
A typo in the YAML will stop the notebook right here, not halfway through the bootstrap.

`uc_bootstrap` requires PyYAML (preinstalled on Databricks). If you still see
`ModuleNotFoundError: No module named 'yaml'`, add a `%pip install pyyaml` cell at the very top of the notebook and run it from the beginning.

In [0]:
import os
import sys

REPO = os.getcwd()                          # notebook folder = repo root folder
SIM = os.path.join(REPO, "retail-simulator")

# Quick diagnostics instead of a cryptic ModuleNotFoundError further down.
# If any assert fails: the notebook is not in the repo root folder
# or the Git folder does not have the latest commit (do a Pull).
assert os.path.isfile(os.path.join(REPO, "uc_bootstrap.py")), f"uc_bootstrap.py missing in {REPO}"
assert os.path.isfile(os.path.join(SIM, "simulator", "cli.py")), f"simulator/cli.py missing in {SIM}"
assert os.path.isfile(os.path.join(REPO, "config", "platform.yml")), f"config/platform.yml missing in {REPO}"

# "if not in" prevents appending the same path every time the cell is run.
for path in (REPO, SIM):
    if path not in sys.path:
        sys.path.insert(0, path)

# Both libraries have a function called main, so we give them different names (aliases).
from uc_bootstrap import main as uc_main      # creates catalogs / schemas / volumes
from uc_bootstrap import load_config          # loads config/platform.yml
from simulator.cli import main as sim_main    # generates the source files

# Same file and same validation as in uc_bootstrap.py - catalog names live in one place.
CONFIG = load_config()

# Where the simulator writes files. Under this path are the per-system volumes (erp, crm, ecom, pos)
# created in step 4. The simulator adds on its own: <system>/<table>/load_date=YYYY-MM-DD/<file>
LANDING = CONFIG.landing_path

# Temporary folder for the smoke test (step 6). It lives in a volume because Spark on serverless
# CANNOT read the driver's local /tmp (LocalFilesystemAccessDeniedException).
# The folder is deleted at the end of the test.
SMOKE = f"{CONFIG.checkpoints_path('dev')}/_smoke"

print("Repo:   ", REPO)
print("Project:", CONFIG.project)
print("Landing:", LANDING)

## 3. SQL plan (dry-run)

`uc_main([...])` takes the same arguments as `python uc_bootstrap.py ...` in a terminal, just as a list of strings.
An empty list or a list without flags is a regular run. **Always pass the list explicitly**: without it argparse would read `sys.argv`, i.e. the notebook process arguments, and could crash.

`--dry-run` only **prints** 24 SQL statements. It executes nothing. It is the same output you saw locally in the terminal.

In [0]:
uc_main(["--dry-run"])

## 4. Unity Catalog bootstrap

Executes the statements from step 3 via `spark.sql`, in order: parent before child (catalog → schema → volume).
Result:
```
<project>_sources.landing            volumes: erp, crm, ecom, pos   ← "outside world", written only by the simulator
<project>_dev / <project>_prod       bronze_erp, bronze_crm, bronze_ecom, bronze_pos, silver, gold,
                                     ops (volume: checkpoints)
```
`<project>` is the `project` value from `config/platform.yml` (e.g. `moustache_markt`).

If `CREATE CATALOG` throws an error about *managed location / storage*, create the three catalogs manually in **Catalog Explorer**:
`<project>_sources`, `<project>_dev`, `<project>_prod`. Then run this cell again. `IF NOT EXISTS` will skip the existing catalogs and add the rest.

In [0]:
uc_main([])

## 5. UC verification

`display(spark.sql(...))` instead of a `%sql` cell: a `%sql` cell with two queries shows only the result of the last one.

In [0]:
display(spark.sql(f"SHOW VOLUMES IN {CONFIG.sources_catalog}.landing"))   # expected: crm, ecom, erp, pos
display(spark.sql(f"SHOW SCHEMAS IN {CONFIG.env_catalog('dev')}"))         # expected: bronze_*, silver, gold, ops (+ default, information_schema)

## 6. Parquet write smoke test

Generates 2 days of ERP only at a small scale (`--systems erp --scale 0.05`) into a temporary folder and reads it with Spark.
It checks two things before you launch a several-minute backfill:
- whether the simulator can write Parquet (pyarrow),
- whether it can write to a volume.

The `_smoke` folder lives in `<project>_dev.ops.checkpoints`, not in landing, so as not to clutter the source data.
`try/finally` deletes it even if the read fails.

In [0]:
try:
    sim_main(["backfill", "--start", "2025-01-01", "--end", "2025-01-02",
              "--systems", "erp", "--scale", "0.05", "--output", SMOKE])

    # Spark recognizes the load_date=... folder as the load_date partition column on its own.
    spark.read.parquet(f"{SMOKE}/erp/product").show(5, truncate=False)
finally:
    dbutils.fs.rm(SMOKE, recurse=True)   # cleanup, regardless of the outcome

## 7. History backfill into landing

Generates all systems (ERP, CRM, e-com, POS) for the whole range from step 1.
Every 30 days the simulator prints progress, and at the end a summary: rows per table, number of files, MB.

- **No type corruption** (`"$ 123"`): it only appears in `daily` mode. If it were in the history, schema inference in episode 1 would silently turn those columns into STRING.
- **Guard:** if landing already has any files, the cell does nothing, so as not to burn quota. To force it: `FORCE_BACKFILL = True` in step 1.

In [0]:
def landing_has_files() -> bool:
    """True if any volume in landing contains at least one file/folder."""
    return any(os.listdir(os.path.join(LANDING, system)) for system in os.listdir(LANDING))

if landing_has_files() and not FORCE_BACKFILL:
    print(f"Landing already has data - skipping backfill. To force it: FORCE_BACKFILL = True in step 1.")
else:
    sim_main(["backfill", "--start", BACKFILL_START, "--end", BACKFILL_END,
              "--scale", str(SCALE), "--output", LANDING])

## 8. Landing verification

For each table: number of days (`load_date=...`), files, MB and date range.
What to expect:
- ERP: 1 file per day for each of the 3 tables,
- POS has **fewer days** than the rest, because stores are closed on Easter Sunday, Thanksgiving and Christmas Day and there is no file on those days (an intentional trap).

`os.walk` walks the volume folders recursively, like a regular disk: volumes are mounted under `/Volumes`.

In [0]:
rows = []
for system in sorted(os.listdir(LANDING)):
    for table in sorted(os.listdir(os.path.join(LANDING, system))):
        table_dir = os.path.join(LANDING, system, table)
        days = sorted(d.removeprefix("load_date=") for d in os.listdir(table_dir))
        files = [os.path.join(root, f) for root, _, names in os.walk(table_dir) for f in names]
        rows.append((system, table, len(days), len(files),
                     round(sum(os.path.getsize(f) for f in files) / 1e6, 1),
                     days[0] if days else None, days[-1] if days else None))

display(spark.createDataFrame(rows, "system string, table string, days int, files int, mb double, first_day string, last_day string"))